In [1]:
import pandas as pd
import numpy as np
import pickle

################################
# Load the data
################################
data = pd.read_pickle("./data/a5_q1.pkl")

y_train = data["y_train"]
X_train_original = data["X_train"]  # Original dataset
X_train_ohe = data["X_train_ohe"]  # One-hot-encoded dataset

X_test_original = data["X_test"]
X_test_ohe = data["X_test_ohe"]

################################
# Produce submission
################################


def create_submission(confidence_scores, save_path):
    """Creates an output file of submissions for Kaggle

    Parameters
    ----------
    confidence_scores : list or numpy array
        Confidence scores (from predict_proba methods from classifiers) or
        binary predictions (only recommended in cases when predict_proba is
        not available)
    save_path : string
        File path for where to save the submission file.

    Example:
    create_submission(my_confidence_scores, './data/submission.csv')

    """
    import pandas as pd

    submission = pd.DataFrame({"score": confidence_scores})
    submission.to_csv(save_path, index_label="id")

In [2]:
from sklearn.model_selection import train_test_split

X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train_ohe, y_train, test_size=0.2, random_state=42
)

In [3]:
# Let's check the shapes of our datasets to confirm that the preprocessing steps were successful.
print("Training set shape:", X_train_final.shape)
print("Validation set shape:", X_val.shape)
print("Test set shape:", X_test_ohe.shape)

# NORMALIZATION
# We will apply standardization to the numerical features in our dataset. However, since we are using the one-hot-encoded version of the dataset, we need to identify which features are numerical and which are categorical. The one-hot-encoded features are already binary (0 or 1), so we will only apply standardization to the original numerical features in the dataset (X_train_original and X_test_original).
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(
    X_train_original.select_dtypes(include=[np.number]).fillna(0)
)
X_test_scaled = scaler.transform(
    X_test_original.select_dtypes(include=[np.number]).fillna(0)
)
# Now we have the scaled versions of the original datasets, we can combine them with the one-hot-encoded features to create our final training and test datasets for classification.
X_train_final = np.hstack((X_train_scaled, X_train_ohe))
X_test_final = np.hstack((X_test_scaled, X_test_ohe))
# Let's check the shapes of our final datasets to confirm that the preprocessing steps were successful.
print("Final training set shape:", X_train_final.shape)
print("Final test set shape:", X_test_final.shape)

Training set shape: (76409, 940)
Validation set shape: (19103, 940)
Test set shape: (23878, 940)
Final training set shape: (95512, 959)
Final test set shape: (23878, 959)


In [4]:
# Finally, we will create a validation set from our final training dataset to evaluate our model's performance before making predictions on the test set.
X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train_final, y_train, test_size=0.2, random_state=42
)
print("Final training set shape after split:", X_train_final.shape)
print("Validation set shape after split:", X_val.shape)

Final training set shape after split: (76409, 959)
Validation set shape after split: (19103, 959)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import roc_auc_score
import numpy as np

# 🔥 1. Define parameter grid (THIS is where the magic happens)
param_dist = {
    "n_estimators": [200, 400, 600, 800],  # more trees = better stability
    "max_depth": [5, 10, 20, None],  # control overfitting
    "min_samples_split": [2, 5, 10, 20],  # prevent overly specific splits
    "min_samples_leaf": [1, 2, 4, 8],  # smooth predictions
    "max_features": ["sqrt", "log2", 0.5],  # feature randomness
    "bootstrap": [True, False],
    "class_weight": ["balanced", None],  # 🔥 helps if classes are imbalanced
}

# 🔥 2. Base model
rf = RandomForestClassifier(random_state=42)

# 🔥 3. Randomized search (FASTER than GridSearch, good for Kaggle)
rf_random = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=25,  # number of models to try (increase if you have time)
    scoring="roc_auc",  # VERY IMPORTANT (matches competition metric)
    cv=3,  # cross-validation for stability
    verbose=2,
    random_state=42,
    n_jobs=-1,
)

# 🔥 4. Fit on training data
rf_random.fit(X_train_final, y_train_final)

# 🔥 5. Best model
best_rf = rf_random.best_estimator_

print("Best Params:", rf_random.best_params_)

# 🔥 6. Evaluate on validation
y_val_pred_proba = best_rf.predict_proba(X_val)[:, 1]
auc_roc = roc_auc_score(y_val, y_val_pred_proba)

print("🔥 Tuned Random Forest AUC-ROC:", auc_roc)

# 🔥 7. Test predictions
y_test_pred_proba = best_rf.predict_proba(X_test_final)[:, 1]

Fitting 3 folds for each of 25 candidates, totalling 75 fits
[CV] END bootstrap=True, class_weight=None, max_depth=5, max_features=log2, min_samples_leaf=2, min_samples_split=20, n_estimators=200; total time=   8.8s
[CV] END bootstrap=True, class_weight=None, max_depth=5, max_features=log2, min_samples_leaf=2, min_samples_split=20, n_estimators=200; total time=   8.8s
[CV] END bootstrap=True, class_weight=None, max_depth=5, max_features=log2, min_samples_leaf=2, min_samples_split=20, n_estimators=200; total time=   8.8s
[CV] END bootstrap=False, class_weight=balanced, max_depth=5, max_features=log2, min_samples_leaf=4, min_samples_split=5, n_estimators=600; total time=  27.3s
[CV] END bootstrap=False, class_weight=balanced, max_depth=5, max_features=log2, min_samples_leaf=4, min_samples_split=5, n_estimators=600; total time=  27.5s
[CV] END bootstrap=False, class_weight=balanced, max_depth=5, max_features=log2, min_samples_leaf=4, min_samples_split=5, n_estimators=600; total time=  28.

/Users/arushisingh/miniconda3/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV] END bootstrap=False, class_weight=None, max_depth=None, max_features=sqrt, min_samples_leaf=4, min_samples_split=5, n_estimators=800; total time= 5.0min
[CV] END bootstrap=False, class_weight=balanced, max_depth=None, max_features=sqrt, min_samples_leaf=2, min_samples_split=5, n_estimators=800; total time= 5.8min
[CV] END bootstrap=False, class_weight=balanced, max_depth=None, max_features=sqrt, min_samples_leaf=2, min_samples_split=5, n_estimators=800; total time= 5.8min
[CV] END bootstrap=True, class_weight=balanced, max_depth=10, max_features=0.5, min_samples_leaf=1, min_samples_split=10, n_estimators=600; total time=14.5min
[CV] END bootstrap=True, class_weight=balanced, max_depth=10, max_features=0.5, min_samples_leaf=1, min_samples_split=10, n_estimators=600; total time=14.5min
[CV] END bootstrap=False, class_weight=None, max_depth=None, max_features=sqrt, min_samples_leaf=4, min_samples_split=5, n_estimators=800; total time= 6.7min
[CV] END bootstrap=True, class_weight=bala

In [ ]:
# Create submission file
create_submission(y_test_pred_proba, "./data/rf_submission.csv")

In [ ]:
from xgboost import XGBClassifier

# Assuming you already have:
# best_rf

# Train an XGBoost model
xgb_model = XGBClassifier(
    random_state=42, use_label_encoder=False, eval_metric="logloss"
)
xgb_model.fit(X_train_final, y_train_final)

rf_val = best_rf.predict_proba(X_val)[:, 1]
xgb_val = xgb_model.predict_proba(X_val)[:, 1]

# Try multiple blends FAST
for w in [0.2, 0.3, 0.4, 0.5, 0.6]:
    blend = w * rf_val + (1 - w) * xgb_val
    auc = roc_auc_score(y_val, blend)
    print(f"RF weight {w}: AUC = {auc:.6f}")

/Users/arushisingh/miniconda3/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [20:53:42] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


RF weight 0.2: AUC = 0.954631
RF weight 0.3: AUC = 0.956150
RF weight 0.4: AUC = 0.957273
RF weight 0.5: AUC = 0.958038
RF weight 0.6: AUC = 0.958448


In [ ]:
# Save the xgb model submission file
y_test_pred_proba_xgb = xgb_model.predict_proba(X_test_final)[:, 1]
create_submission(y_test_pred_proba_xgb, "./data/xgb_submission.csv")

In [11]:
rf_test = best_rf.predict_proba(X_test_final)[:, 1]
xgb_test = xgb_model.predict_proba(X_test_final)[:, 1]

best_w = 0.6  # replace with best from above
final_pred = best_w * rf_test + (1 - best_w) * xgb_test

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from scipy.stats import rankdata

# =========================================================
# 1. TRAIN A STRONG RANDOM FOREST
# =========================================================
# Changes made:
# - increased number of trees for more stable predictions
# - controlled tree depth to reduce overfitting
# - used min_samples_split and min_samples_leaf for regularization
# - used class_weight='balanced' in case classes are imbalanced

rf_model = RandomForestClassifier(
    n_estimators=800,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

rf_model.fit(X_train_final, y_train_final)

# Validation predictions
rf_val_pred = rf_model.predict_proba(X_val)[:, 1]
rf_auc = roc_auc_score(y_val, rf_val_pred)
print(f"Random Forest Validation AUC: {rf_auc:.6f}")


# =========================================================
# 2. TRAIN A STRONG XGBOOST
# =========================================================
# Changes made:
# - switched to boosting model for stronger tabular performance
# - used low learning rate + many estimators
# - used subsampling and column sampling for better generalization
# - used regularization to reduce overfitting
# - used early stopping based on validation AUC

xgb_model = XGBClassifier(
    n_estimators=3000,
    learning_rate=0.015,
    max_depth=4,
    min_child_weight=3,
    gamma=0.2,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=2.0,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=100,
)

xgb_model.fit(X_train_final, y_train_final, eval_set=[(X_val, y_val)], verbose=False)

# Validation predictions
xgb_val_pred = xgb_model.predict_proba(X_val)[:, 1]
xgb_auc = roc_auc_score(y_val, xgb_val_pred)
print(f"XGBoost Validation AUC: {xgb_auc:.6f}")


# =========================================================
# 3. TEST PROBABILITY BLENDS
# =========================================================
# We try weighted averages of RF and XGB probabilities.
# This often improves leaderboard performance because the
# two models make different errors.

print("\nProbability Blend Results:")
best_blend_auc = -1
best_weight = None

for w in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    blend_val_pred = w * rf_val_pred + (1 - w) * xgb_val_pred
    blend_auc = roc_auc_score(y_val, blend_val_pred)
    print(f"RF weight = {w:.1f}, XGB weight = {1-w:.1f} --> AUC = {blend_auc:.6f}")

    if blend_auc > best_blend_auc:
        best_blend_auc = blend_auc
        best_weight = w

print(f"\nBest probability blend AUC: {best_blend_auc:.6f}")
print(f"Best RF weight: {best_weight:.1f}")
print(f"Best XGB weight: {1-best_weight:.1f}")


# =========================================================
# 4. OPTIONAL: TEST RANK BLENDING TOO
# =========================================================
# Since AUC depends on ranking, rank averaging can sometimes
# beat probability averaging.

rf_val_rank = rankdata(rf_val_pred)
xgb_val_rank = rankdata(xgb_val_pred)

print("\nRank Blend Results:")
best_rank_auc = -1
best_rank_weight = None

for w in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    rank_blend_val = w * rf_val_rank + (1 - w) * xgb_val_rank
    rank_auc = roc_auc_score(y_val, rank_blend_val)
    print(
        f"RF rank weight = {w:.1f}, XGB rank weight = {1-w:.1f} --> AUC = {rank_auc:.6f}"
    )

    if rank_auc > best_rank_auc:
        best_rank_auc = rank_auc
        best_rank_weight = w

print(f"\nBest rank blend AUC: {best_rank_auc:.6f}")
print(f"Best RF rank weight: {best_rank_weight:.1f}")
print(f"Best XGB rank weight: {1-best_rank_weight:.1f}")


# =========================================================
# 5. CREATE TEST PREDICTIONS USING THE BETTER BLEND METHOD
# =========================================================

rf_test_pred = rf_model.predict_proba(X_test_final)[:, 1]
xgb_test_pred = xgb_model.predict_proba(X_test_final)[:, 1]

if best_rank_auc > best_blend_auc:
    print("\nUsing RANK blend for final submission.")
    rf_test_rank = rankdata(rf_test_pred)
    xgb_test_rank = rankdata(xgb_test_pred)
    final_test_pred = (
        best_rank_weight * rf_test_rank + (1 - best_rank_weight) * xgb_test_rank
    )
else:
    print("\nUsing PROBABILITY blend for final submission.")
    final_test_pred = best_weight * rf_test_pred + (1 - best_weight) * xgb_test_pred


# =========================================================
# 6. SAVE SUBMISSION
# =========================================================
# IMPORTANT:
# Replace 'id' and 'target' below with the exact column names
# required by your Kaggle competition.

submission = pd.DataFrame(
    {
        "id": test_ids,  # replace with your actual ID variable if needed
        "target": final_test_pred,  # replace with correct target column name if needed
    }
)

submission.to_csv("submission_blend.csv", index=False)
print("\nSaved file: submission_blend.csv")

Random Forest Validation AUC: 0.940926


KeyboardInterrupt: 